# Chapter 7 &mdash; Theorem: $L$ is Regular iff Some NFA Recognizes It

**Concept 9 of the Chapter 7 decomposition:** *Theorem: $L$ is Regular iff Some NFA Recognizes It*

Both directions in two lines &mdash; every DFA is an NFA, and subset construction converts back.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Regular-Iff-NFA/Concept-Regular-Iff-NFA.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateNFA     import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateNFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateNFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


> **Theorem.** $L$ is regular $\iff$ some NFA recognizes $L$.

**($\Rightarrow$)** A DFA **is** an NFA: take $Q_0=\{q_0\}$ and read each
$\delta(q,a)=q'$ as $\{q'\}$. Nothing to prove.

**($\Leftarrow$)** Given an NFA, the subset construction yields a DFA for the same
language. Concept 8 is the proof.

Two lines each &mdash; and the payoff is large: from here on you may design with
whichever model is convenient and convert when you need the other. Chapters 8&ndash;10 lean
on this constantly.

## 2. Definitions

### Direction 1: a DFA, viewed as an NFA

In [ ]:
D = md2mc('''DFA
IF : 0 -> Od
IF : 1 -> IF
Od : 0 -> IF
Od : 1 -> Od
''')

def dfa_as_nfa(D):
    Dl = {(q, a): {t} for (q, a), t in D["Delta"].items()}
    return mk_nfa(D["Q"], D["Sigma"], Dl, {D["q0"]}, D["F"])

### Direction 2: an NFA, converted by subset construction

In [ ]:
N = md2mc('''NFA
I : 0 | 1 -> I
I : 1 -> A
A : 0 | 1 -> F
''')

<!-- nav-strip -->

---

&larr;&nbsp;[Ch7&nbsp;8.&nbsp;Subset Construction: Converting an NFA to a DFA](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Subset-Construction/Concept-Subset-Construction.ipynb) &nbsp;&middot;&nbsp; [**Chapter 7** index](https://github.com/ganeshutah/Jove/blob/master/Chapter7/README.md) &nbsp;&middot;&nbsp; [Ch7&nbsp;10.&nbsp;Brzozowski's Minimization: Reverse, Determinize, Reverse, Determinize](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Brzozowski-Minimization/Concept-Brzozowski-Minimization.ipynb)&nbsp;&rarr;

---

## 3. Tests

Direction 1: the reading is trivially faithful.

In [ ]:
As_nfa = dfa_as_nfa(D)
from itertools import product
strs = [''.join(p) for k in range(11) for p in product('01', repeat=k)]
assert all(accepts_nfa(As_nfa, s) == accepts_dfa(D, s) for s in strs)
print("DFA-as-NFA accepts exactly what the DFA does, on all %d strings" % len(strs))
print("|Q0| =", len(As_nfa["Q0"]), " -- a singleton, which is what makes it deterministic")

Direction 2: `nfa2dfa` closes the loop.

In [ ]:
Back = nfa2dfa(N)
assert all(accepts_dfa(Back, s) == accepts_nfa(N, s) for s in strs)
print("subset construction preserves the language on all %d strings" % len(strs))

Round trip: DFA &rarr; NFA &rarr; DFA returns an isomorphic minimal machine.

In [ ]:
round_trip = min_dfa(nfa2dfa(dfa_as_nfa(D)))
print("original minimal %d states, round-tripped %d states"
      % (len(min_dfa(D)["Q"]), len(round_trip["Q"])))
assert iso_dfa(min_dfa(D), round_trip)
print("isomorphic? ", iso_dfa(min_dfa(D), round_trip))

So the two models describe **exactly** the same class of languages.

In [ ]:
print("regular languages  ==  DFA languages  ==  NFA languages")
print("\nChapters 8-10 add regular expressions to that chain.")

## 4. Animation

The NFA and its determinization recognise one language; here is the DFA.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa(nfa2dfa(N)), FuseEdges=True)

## 5. Exercises


1. Write out direction ($\Rightarrow$) as a formal proof. How long is it really?
2. Does the round trip ever *grow* the minimal DFA? Why not?
3. What would change if NFA acceptance were "**all** copies land in $F$"?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter7/Concept-Regular-Iff-NFA')